# Camada Bronze Ingestão de Dados Brutos

**Pipeline:** Análise da Alfabetização no Brasil
**Responsabilidade:** ingerir dados brutos (batch + streaming) e disponibilizá-los em Parquet para a camada Silver.

**Fontes:**
- **Batch:** Base dos Dados + Microdados INEP (CSV)
- **Streaming:** eventos simulados via Azure Event Hubs (JSON)

**Princípio da camada:** a Bronze não transforma dados — apenas ingere e padroniza o formato de saída em Parquet. Nenhuma limpeza acontece aqui.

## 1. Configuração de Acesso

Conexão ao ADLS Gen2 via **Azure Key Vault** — a chave de acesso é lida em runtime e nunca fica exposta no código (boa prática de segurança).

In [1]:
# ============================================================
# Notebook Bronze — Ingestão de dados brutos
# Responsabilidade: ler fontes e disponibilizar na camada Bronze
# ============================================================
from notebookutils import mssparkutils

CONTA_STORAGE = "stalfalfabetizacao"

CHAVE_ACESSO = mssparkutils.credentials.getSecret(
    "https://kv-alfabetizacao.vault.azure.net/",
    "storage-account-key"
)

spark.conf.set(
    f"fs.azure.account.key.{CONTA_STORAGE}.dfs.core.windows.net",
    CHAVE_ACESSO
)

CAMINHO_BRONZE = f"abfss://bronze@{CONTA_STORAGE}.dfs.core.windows.net/"

print("Conexão configurada com sucesso!")
print(f"  Bronze: {CAMINHO_BRONZE}")

StatementMeta(sparkpool, 7, 2, Finished, Available, Finished, False)

Conexão configurada com sucesso!
  Bronze: abfss://bronze@stalfalfabetizacao.dfs.core.windows.net/


## 2. Ingestão Batch — Leitura dos CSVs

Leitura das fontes históricas. Sem transformação — apenas confirma que os arquivos estão acessíveis e conta as linhas.

In [2]:
# ============================================================
# Ingestão Batch — leitura dos CSVs brutos da Bronze
# Sem transformação — só confirma que os arquivos estão lá
# ============================================================

# Fontes Base dos Dados (separador vírgula)
df_uf            = spark.read.csv(CAMINHO_BRONZE + "uf.csv",                header=True, inferSchema=True)
df_municipio     = spark.read.csv(CAMINHO_BRONZE + "municipio.csv",         header=True, inferSchema=True)
df_meta_brasil   = spark.read.csv(CAMINHO_BRONZE + "meta_brasil.csv",       header=True, inferSchema=True)
df_meta_uf       = spark.read.csv(CAMINHO_BRONZE + "meta_por_uf.csv",       header=True, inferSchema=True)
df_meta_municipio= spark.read.csv(CAMINHO_BRONZE + "meta_por_municipio.csv",header=True, inferSchema=True)

# Fontes INEP (separador ponto-e-vírgula)
df_ts_municipio  = spark.read.csv(CAMINHO_BRONZE + "TS_MUNICIPIO.csv", header=True, inferSchema=True, sep=";")
df_ts_estado     = spark.read.csv(CAMINHO_BRONZE + "TS_ESTADO.csv",    header=True, inferSchema=True, sep=";")
df_ts_item       = spark.read.csv(CAMINHO_BRONZE + "TS_ITEM.csv",      header=True, inferSchema=True, sep=";")

print("Batch Bronze lido:")
print(f"  uf:               {df_uf.count()} linhas")
print(f"  municipio:        {df_municipio.count()} linhas")
print(f"  meta_brasil:      {df_meta_brasil.count()} linhas")
print(f"  meta_uf:          {df_meta_uf.count()} linhas")
print(f"  meta_municipio:   {df_meta_municipio.count()} linhas")
print(f"  ts_municipio:     {df_ts_municipio.count()} linhas")
print(f"  ts_estado:        {df_ts_estado.count()} linhas")
print(f"  ts_item:          {df_ts_item.count()} linhas")

StatementMeta(sparkpool, 7, 3, Finished, Available, Finished, False)

Batch Bronze lido:
  uf:               145 linhas
  municipio:        23995 linhas
  meta_brasil:      3 linhas
  meta_uf:          54 linhas
  meta_municipio:   10704 linhas
  ts_municipio:     12416 linhas
  ts_estado:        80 linhas
  ts_item:          1960 linhas


## 3. Gravação Batch em Parquet

Converte os CSVs para **Parquet** (colunar + comprimido). Padroniza a saída: a Silver sempre lê Parquet, sem saber a origem. Menor custo de storage e queries mais rápidas — FinOps na prática.

In [3]:
# ============================================================
# Gravação do batch em Parquet na Bronze
# Padroniza o formato de saída — Silver sempre lê Parquet
# ============================================================

df_uf.write.mode("overwrite").parquet(CAMINHO_BRONZE + "uf/")
df_municipio.write.mode("overwrite").parquet(CAMINHO_BRONZE + "municipio/")
df_meta_brasil.write.mode("overwrite").parquet(CAMINHO_BRONZE + "meta_brasil/")
df_meta_uf.write.mode("overwrite").parquet(CAMINHO_BRONZE + "meta_por_uf/")
df_meta_municipio.write.mode("overwrite").parquet(CAMINHO_BRONZE + "meta_por_municipio/")
df_ts_municipio.write.mode("overwrite").parquet(CAMINHO_BRONZE + "ts_municipio/")
df_ts_estado.write.mode("overwrite").parquet(CAMINHO_BRONZE + "ts_estado/")
df_ts_item.write.mode("overwrite").parquet(CAMINHO_BRONZE + "ts_item/")

print("Batch gravado em Parquet na Bronze:")
print(f"  {CAMINHO_BRONZE}uf/")
print(f"  {CAMINHO_BRONZE}municipio/")
print(f"  {CAMINHO_BRONZE}meta_brasil/")
print(f"  {CAMINHO_BRONZE}meta_por_uf/")
print(f"  {CAMINHO_BRONZE}meta_por_municipio/")
print(f"  {CAMINHO_BRONZE}ts_municipio/")
print(f"  {CAMINHO_BRONZE}ts_estado/")
print(f"  {CAMINHO_BRONZE}ts_item/")

StatementMeta(sparkpool, 7, 4, Finished, Available, Finished, False)

Batch gravado em Parquet na Bronze:
  abfss://bronze@stalfalfabetizacao.dfs.core.windows.net/uf/
  abfss://bronze@stalfalfabetizacao.dfs.core.windows.net/municipio/
  abfss://bronze@stalfalfabetizacao.dfs.core.windows.net/meta_brasil/
  abfss://bronze@stalfalfabetizacao.dfs.core.windows.net/meta_por_uf/
  abfss://bronze@stalfalfabetizacao.dfs.core.windows.net/meta_por_municipio/
  abfss://bronze@stalfalfabetizacao.dfs.core.windows.net/ts_municipio/
  abfss://bronze@stalfalfabetizacao.dfs.core.windows.net/ts_estado/
  abfss://bronze@stalfalfabetizacao.dfs.core.windows.net/ts_item/


## 4. Ingestão Streaming → Parquet

Leitura dos eventos JSON depositados pelo consumer (`streaming_ingest/`). O **schema é definido manualmente** para evitar erro de inferência (`_corrupt_record`) e garantir tipagem consistente. Grava em Parquet separado do batch.

In [4]:
# ============================================================
# Gravação dos eventos de streaming na Bronze em Parquet
# Schema definido manualmente baseado nos JSONs do produtor
# ============================================================
from pyspark.sql import functions as F
from pyspark.sql.types import *

# Caminho dos JSONs de streaming gravados pelo consumer
CAMINHO_STREAMING = CAMINHO_BRONZE + "streaming_ingest/"

# Schemas baseados nos JSONs reais do produtor
schema_aluno = StructType([
    StructField("tipo_mensagem",    StringType()),
    StructField("NU_ANO",           IntegerType()),
    StructField("CO_UF",            IntegerType()),
    StructField("SG_UF",            StringType()),
    StructField("ID_ALUNO",         LongType()),
    StructField("TP_SERIE",         IntegerType()),
    StructField("ID_ESCOLA",        LongType()),
    StructField("TP_DEPENDENCIA",   IntegerType()),
    StructField("CO_MUNICIPIO",     LongType()),
    StructField("NO_MUNICIPIO",     StringType()),
    StructField("IN_PRESENCA",      IntegerType()),
    StructField("VL_PROFICIENCIA",  DoubleType()),
    StructField("IN_ALFABETIZADO",  IntegerType()),
    StructField("timestamp_evento", StringType()),
])

schema_municipio = StructType([
    StructField("tipo_mensagem",    StringType()),
    StructField("NU_ANO",           IntegerType()),
    StructField("CO_UF",            IntegerType()),
    StructField("SG_UF",            StringType()),
    StructField("CO_MUNICIPIO",     LongType()),
    StructField("NO_MUNICIPIO",     StringType()),
    StructField("TP_SERIE",         IntegerType()),
    StructField("VL_MEDIA",         DoubleType()),
    StructField("timestamp_evento", StringType()),
])

schema_indicadores = StructType([
    StructField("tipo_mensagem",       StringType()),
    StructField("NU_ANO",              IntegerType()),
    StructField("CO_UF",               IntegerType()),
    StructField("SG_UF",               StringType()),
    StructField("TP_SERIE",            IntegerType()),
    StructField("ID_TIPO_REDE",        IntegerType()),
    StructField("PC_ALUNO_MEDIA",      DoubleType()),
    StructField("PC_ALUNO_NIVEL_8_LP", DoubleType()),
    StructField("timestamp_evento",    StringType()),
])

schema_itens = StructType([
    StructField("tipo_mensagem",    StringType()),
    StructField("NU_ANO",           IntegerType()),
    StructField("CO_UF",            IntegerType()),
    StructField("SG_UF",            StringType()),
    StructField("CO_BLOCO",         StringType()),
    StructField("NU_POSICAO",       IntegerType()),
    StructField("CO_ITEM",          IntegerType()),
    StructField("TP_DISCIPLINA",    IntegerType()),
    StructField("NU_DESCR_ITEM",    StringType()),
    StructField("TX_GABARITO",      StringType()),
    StructField("NU_PARAM_A1",      DoubleType()),
    StructField("NU_PARAM_B1",      DoubleType()),
    StructField("IN_ITEM_COMUM",    IntegerType()),
    StructField("timestamp_evento", StringType()),
])

# Lê com schema definido e grava em Parquet
def ler_e_gravar(pasta, schema, nome_saida):
    caminho_entrada = CAMINHO_STREAMING + f"tipo_mensagem={pasta}/"
    caminho_saida   = CAMINHO_BRONZE + f"parquet/{nome_saida}/"
    df = spark.read.schema(schema).json(caminho_entrada)
    df.write.mode("overwrite").parquet(caminho_saida)
    print(f"  {nome_saida}: {df.count()} eventos → {caminho_saida}")

print("Gravando streaming Bronze em Parquet:")
ler_e_gravar("dados_aluno",        schema_aluno,        "stream_aluno")
ler_e_gravar("dados_municipio",    schema_municipio,    "stream_municipio")
ler_e_gravar("dados_indicadores",  schema_indicadores,  "stream_indicadores")
ler_e_gravar("dados_itens",        schema_itens,        "stream_itens")
print("\nBronze streaming gravada com sucesso!")

StatementMeta(sparkpool, 7, 5, Finished, Available, Finished, False)

Gravando streaming Bronze em Parquet:
  stream_aluno: 16 eventos → abfss://bronze@stalfalfabetizacao.dfs.core.windows.net/parquet/stream_aluno/
  stream_municipio: 11 eventos → abfss://bronze@stalfalfabetizacao.dfs.core.windows.net/parquet/stream_municipio/
  stream_indicadores: 11 eventos → abfss://bronze@stalfalfabetizacao.dfs.core.windows.net/parquet/stream_indicadores/
  stream_itens: 16 eventos → abfss://bronze@stalfalfabetizacao.dfs.core.windows.net/parquet/stream_itens/

Bronze streaming gravada com sucesso!
